# Neo4j UC Federation Prototype: Validation Test Suite

## Overview

This notebook validates Neo4j connectivity through Unity Catalog's generic JDBC path (`TYPE JDBC`). It provides a systematic test progression from basic network connectivity through the full UC JDBC connection.

All tests ran on a live Databricks cluster connected to Neo4j Aura.

## Pre-requisite: Populate Neo4j

Before running this notebook, execute the following Cypher in your Neo4j Browser or `cypher-shell` to create the sample data:

```cypher
CREATE (a1:Aircraft {aircraft_id: 'AC-001', tail_number: 'N12345', model: '737-800', manufacturer: 'Boeing', operator: 'SkyWest'})
CREATE (a2:Aircraft {aircraft_id: 'AC-002', tail_number: 'N67890', model: 'A320neo', manufacturer: 'Airbus', operator: 'Delta'})
CREATE (c1:Component {component_id: 'CMP-001', name: 'CFM56-7B Engine', category: 'Engine', status: 'Active'})
CREATE (a1)-[:HAS_COMPONENT {installed_date: '2024-01-15', position: 'left-wing'}]->(c1)
CREATE (a2)-[:HAS_COMPONENT {installed_date: '2024-06-01', position: 'right-wing'}]->(c1)
```

---

## Configuration

Replace the placeholder values below with your Neo4j and Databricks environment details.

In [ ]:
# =============================================================================
# CONFIGURATION - Replace placeholders with your values
# =============================================================================

NEO4J_HOST = "<YOUR_NEO4J_HOST>"          # e.g. "abc123.databases.neo4j.io"
NEO4J_USER = "<YOUR_NEO4J_USER>"          # e.g. "neo4j"
NEO4J_PASSWORD = "<YOUR_NEO4J_PASSWORD>"
NEO4J_DATABASE = "<YOUR_NEO4J_DATABASE>"  # e.g. "neo4j"

JDBC_JAR_PATH = "<YOUR_JDBC_JAR_UC_VOLUME_PATH>"  # e.g. "/Volumes/catalog/schema/jars/neo4j-uc-connector.jar"
UC_CONNECTION_NAME = "<YOUR_UC_CONNECTION_NAME>"   # e.g. "neo4j_jdbc"

# Single JAR dependency for CREATE CONNECTION
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

# Derived URLs
NEO4J_BOLT_URI = f"neo4j+s://{NEO4J_HOST}"
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"

print("Configuration:")
print(f"  Neo4j Host: {NEO4J_HOST}")
print(f"  Bolt URI: {NEO4J_BOLT_URI}")
print(f"  JDBC URL: {NEO4J_JDBC_URL}")
print(f"  Connection Name: {UC_CONNECTION_NAME}")
print(f"  JDBC JAR Path: {JDBC_JAR_PATH}")

---

## Section 1: Environment Information

Capture cluster and runtime details for reproducibility.

In [ ]:
# Collect environment information
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

# Spark version
print(f"\nSpark Version: {spark.version}")

# Databricks Runtime
try:
    dbr_version = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    print(f"Databricks Runtime: {dbr_version}")
except:
    print("Databricks Runtime: Unable to determine")

# Python version
import sys
print(f"Python Version: {sys.version}")

# Check neo4j package
try:
    import neo4j
    print(f"Neo4j Python Driver: {neo4j.__version__}")
except ImportError:
    print("Neo4j Python Driver: NOT INSTALLED")

# Check JAR file exists
print(f"\nJDBC JAR Path: {JDBC_JAR_PATH}")
try:
    files = dbutils.fs.ls(JDBC_JAR_PATH.rsplit('/', 1)[0])
    file_names = [f.name for f in files]
    jdbc_jar_found = JDBC_JAR_PATH.split('/')[-1] in file_names
    print(f"JDBC JAR File Exists: {jdbc_jar_found}")
except Exception as e:
    print(f"JAR File Check Error: {e}")

---

## Section 2: Network Connectivity Test (TCP Layer)

**Expected Result**: PASS - Proves network path is open.

In [ ]:
# TCP connectivity test using netcat
import subprocess
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {NEO4J_HOST}:7687 (Bolt protocol port)")
print("Testing: Can Databricks reach Neo4j at the network level?")

try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', NEO4J_HOST, '7687'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print("\n" + "=" * 60)
        print(">>> CONNECTIVITY VERIFIED <<<")
        print("=" * 60)
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        print(f"\nConnection Details:")
        print(f"  - Host: {NEO4J_HOST}")
        print(f"  - Port: 7687 (Bolt)")
        print(f"  - TCP Latency: {elapsed:.1f}ms")
        if output:
            print(f"  - Raw Output: {output}")
        print("\n" + "-" * 60)
        print("RESULT: Network path to Neo4j is OPEN")
        print("        Firewall rules allow Bolt protocol traffic")
        print("-" * 60)
        print("\nStatus: PASS")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_HOST}:7687 - check firewall rules")
        print(f"Details: {output}")
        print("\nStatus: FAIL")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")

---

## Section 3: Direct JDBC Read Tests (Without SafeSpark Sandbox)

These tests use the Neo4j JDBC driver directly with Spark to read the Aircraft and Component nodes created in the pre-requisite step, **without** Unity Catalog's SafeSpark wrapper.

**Note**: Requires the JDBC JAR to be installed as a cluster library (not just in a UC Volume).

**Schema Inference Issue**: When using `dbtable` option, Spark's schema inference returns `NullType()` for all columns from Neo4j JDBC. **Fix**: Use `customSchema` option to explicitly specify column types.

In [ ]:
# Direct JDBC - Read Aircraft nodes using dbtable option
import time

print("=" * 60)
print("TEST: Direct JDBC - Read Aircraft nodes (dbtable)")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Spark read Aircraft nodes via JDBC using dbtable option?")

AIRCRAFT_SCHEMA = "aircraft_id STRING, tail_number STRING, model STRING, manufacturer STRING, operator STRING"

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("dbtable", "Aircraft") \
        .option("customSchema", AIRCRAFT_SCHEMA) \
        .load()

    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    row_count = len(results)

    print(f"\n[PASS] Read {row_count} Aircraft nodes in {total_time:.1f}ms")
    df.show(truncate=False)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Direct JDBC dbtable read failed: {e}")
    print("\nStatus: FAIL")

In [ ]:
# Direct JDBC - Read Component nodes using dbtable option
import time

print("=" * 60)
print("TEST: Direct JDBC - Read Component nodes (dbtable)")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Spark read Component nodes via JDBC using dbtable option?")

COMPONENT_SCHEMA = "component_id STRING, name STRING, category STRING, status STRING"

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("dbtable", "Component") \
        .option("customSchema", COMPONENT_SCHEMA) \
        .load()

    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    row_count = len(results)

    print(f"\n[PASS] Read {row_count} Component nodes in {total_time:.1f}ms")
    df.show(truncate=False)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Direct JDBC dbtable read failed: {e}")
    print("\nStatus: FAIL")

In [ ]:
# Direct JDBC - SQL Aggregate Query (COUNT)
import time

print("=" * 60)
print("TEST: Direct JDBC - SQL Aggregate (COUNT)")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Neo4j JDBC translate SQL aggregates to Cypher?")
print("SQL Query: SELECT COUNT(*) AS flight_count FROM Flight")
print("Expected Cypher: MATCH (n:Flight) RETURN count(n) AS flight_count")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", "SELECT COUNT(*) AS flight_count FROM Flight") \
        .option("customSchema", "flight_count LONG") \
        .load()

    # Force execution
    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    count = results[0]['flight_count'] if results else 0

    print("\n" + "=" * 60)
    print(">>> SQL AGGREGATE TRANSLATION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] SQL aggregate query executed in {total_time:.1f}ms")
    
    print(f"\nQuery Results:")
    df.show(truncate=False)
    
    print(f"Translation Details:")
    print(f"  - Input SQL: SELECT COUNT(*) AS flight_count FROM Flight")
    print(f"  - Translated to Cypher: MATCH (n:Flight) RETURN count(n)")
    print(f"  - Flight Count: {count:,}")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: SQL Aggregate Translation WORKING")
    print("        COUNT(*) on Neo4j label successfully translated to Cypher")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Direct JDBC aggregate query failed: {e}")
    print("\nStatus: FAIL")
    print("\nTroubleshooting:")
    print("  - Ensure 'Flight' label exists in your Neo4j database")
    print("  - Or change the query to use a label that exists in your graph")

---

## Section 6: Unity Catalog JDBC Connection (Generic TYPE JDBC Path)

This section creates and tests the UC JDBC connection using the generic `TYPE JDBC` connector with a bring-your-own-driver JAR. The SafeSpark sandbox wraps the Neo4j JDBC driver in an isolated JVM, which required memory tuning (see the proposal report for details on the SafeSpark workaround).

**Note:** The `java_dependencies` option in `CREATE CONNECTION` only accepts UC Volume paths. Cluster-installed libraries (Maven coordinates or uploaded JARs) cannot be referenced here — they are a separate system. The JAR must be uploaded to a UC Volume.

In [ ]:
# Create Unity Catalog JDBC Connection
import time

print("=" * 60)
print("SETUP: Create Unity Catalog JDBC Connection")
print("=" * 60)
print(f"\nConnection Name: {UC_CONNECTION_NAME}")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")
print(f"Driver: org.neo4j.jdbc.Neo4jDriver")

# Drop existing connection
spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")
print(f"\n[INFO] Dropped existing connection (if any): {UC_CONNECTION_NAME}")

# Create connection with explicit driver class
# NOTE: customSchema must be in externalOptionsAllowList to bypass Spark schema inference
create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_SQL}',
  user '{NEO4J_USER}',
  password '{NEO4J_PASSWORD}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

print(f"[INFO] Java Dependencies:")
print(f"  - {JDBC_JAR_PATH}")

try:
    start_time = time.time()
    spark.sql(create_sql)
    create_time = (time.time() - start_time) * 1000
    
    print("\n" + "=" * 60)
    print(">>> UC CONNECTION CREATED <<<")
    print("=" * 60)
    print(f"\n[PASS] Connection '{UC_CONNECTION_NAME}' created in {create_time:.1f}ms")
    
    print(f"\nConnection Configuration:")
    print(f"  - Name: {UC_CONNECTION_NAME}")
    print(f"  - Type: JDBC")
    print(f"  - Driver: org.neo4j.jdbc.Neo4jDriver")
    print(f"  - SQL Translation: Enabled")
    
except Exception as e:
    print(f"\n[FAIL] Failed to create connection: {e}")

In [ ]:
# Verify connection configuration
print("=" * 60)
print("VERIFY: Connection Configuration")
print("=" * 60)

try:
    df = spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}")
    print("\nConnection details:")
    df.show(truncate=False)
except Exception as e:
    print(f"\n[FAIL] Cannot describe connection: {e}")

In [ ]:
# Test: COUNT Aggregate
test_uc_query(
    test_name="COUNT Aggregate",
    sql_query="SELECT COUNT(*) AS flight_count FROM Flight",
    custom_schema="flight_count LONG",
    expected_cypher="MATCH (n:Flight) RETURN count(n)"
)